Runs 001_Preprocessing.ipynb for each sample in the list of patients.

In [36]:
import json
import os
from pathlib import Path
import anndata as ad
import pandas as pd
import papermill as pm

### Paths

In [37]:
project  = Path(os.environ.get("CAP_PROJECT","~/Downloads/1105_Project_Nowotny")).expanduser()

In [38]:
input_nb = project / "Nina_notebooks_Aug2026" / "my_version" / "001_Preprocessing.ipynb"
run_dir  = project / "papermill_runs"
head_dir = project / "Adata_Saves" / "head"

In [39]:
run_dir.mkdir(parents=True, exist_ok=True)

### Samples

In [40]:
samples  = pd.read_csv(project / "metadata" / "samples.csv")

In [41]:
sample_ids = samples.loc[samples["include"], "sample_id"].tolist()

### Loop

In [42]:
len(sample_ids)

45

In [43]:
for p in sample_ids:
    print(p)

GSM4909295_ER-AH0319
GSM4909296_ER-MH0001
GSM4909302_ER-MH0025
GSM4909305_ER-MH0029-7C
GSM4909306_ER-MH0029-9C
GSM4909300_ER-MH0032
GSM4909307_ER-MH0040
GSM4909301_ER-MH0042
GSM4909309_ER-MH0043-T
GSM4909311_ER-MH0056-T
GSM4909313_ER-MH0064-T
GSM4909299_ER-MH0114-T3
GSM4909297_ER-MH0125
GSM4909303_ER-MH0151
GSM4909304_ER-MH0163
GSM4909315_ER-MH0167-T
GSM4909317_ER-MH0173-T
GSM4909298_ER-PM0360
GSM4909289_HER2-AH0308
GSM4909291_HER2-MH0031
GSM4909292_HER2-MH0069
GSM4909293_HER2-MH0161
GSM4909294_HER2-MH0176
GSM4909290_HER2-PM0337
GSM4909272_N-MH0021-Total
GSM4909268_N-MH0023-Total
GSM4909263_N-MH0064-Total
GSM4909266_N-MH0169-Total
GSM4909274_N-MH275-Total
GSM4909271_N-MH288-Total
GSM4909254_N-PM0019-Total
GSM4909253_N-PM0092-Total
GSM4909257_N-PM0095-Total
GSM4909261_N-PM0230-Total
GSM4909265_N-PM0233-Total
GSM4909270_N-PM0342-Total
GSM4909276_N-PM0372-Total
GSM4909284_TN-MH0114-T2
GSM4909281_TN-MH0126
GSM4909286_TN-B1-MH0131
GSM4909282_TN-MH0135
GSM4909288_TN-B1-MH0177
GSM4909285_TN-B

In [44]:
import time

In [45]:


def completed(path):
    """papermill writes the output notebook even when a cell raises,
    so existence is not enough - look for stored errors."""
    if not path.exists():
        return False
    nb = json.loads(path.read_text())
    return not any(o.get("output_type") == "error"
                   for c in nb["cells"] for o in c.get("outputs", []))

In [46]:
log = []

In [47]:
for n, patient in enumerate(sample_ids, 1):
    out = run_dir / f"{patient}.ipynb"
    if completed(out):
        print(f"[{n:2d}/{len(sample_ids)}] {patient:32s} skip")
        log.append({"sample_id": patient, "status": "skipped", "seconds": 0, "error": ""})
        continue
    print(f"[{n:2d}/{len(sample_ids)}] {patient:32s} ", end="", flush=True)
    t = time.time()
    try:
        pm.execute_notebook(str(input_nb), str(out),
                            parameters={"patient": patient},
                            kernel_name="cap", progress_bar=False)
        status, err = "ok", ""
        print(f"{time.time() - t:5.0f}s")
    except Exception as e:
        status, err = "failed", str(e)[:500]
        print("FAILED")
    log.append({"sample_id": patient, "status": status,
                "seconds": round(time.time() - t), "error": err})

[ 1/45] GSM4909295_ER-AH0319             skip
[ 2/45] GSM4909296_ER-MH0001             skip
[ 3/45] GSM4909302_ER-MH0025             skip
[ 4/45] GSM4909305_ER-MH0029-7C          skip
[ 5/45] GSM4909306_ER-MH0029-9C          skip
[ 6/45] GSM4909300_ER-MH0032             skip
[ 7/45] GSM4909307_ER-MH0040             skip
[ 8/45] GSM4909301_ER-MH0042             skip
[ 9/45] GSM4909309_ER-MH0043-T           skip
[10/45] GSM4909311_ER-MH0056-T           skip
[11/45] GSM4909313_ER-MH0064-T           skip
[12/45] GSM4909299_ER-MH0114-T3          skip
[13/45] GSM4909297_ER-MH0125             skip
[14/45] GSM4909303_ER-MH0151             skip
[15/45] GSM4909304_ER-MH0163             skip
[16/45] GSM4909315_ER-MH0167-T           skip
[17/45] GSM4909317_ER-MH0173-T           skip
[18/45] GSM4909298_ER-PM0360             skip
[19/45] GSM4909289_HER2-AH0308           skip
[20/45] GSM4909291_HER2-MH0031           skip
[21/45] GSM4909292_HER2-MH0069           skip
[22/45] GSM4909293_HER2-MH0161    

In [48]:
log = pd.DataFrame(log)

In [49]:
log.to_csv(project / "metadata" / "papermill_log.csv", index=False)

In [50]:
assert (log["status"] != "failed").all(), log[log["status"] == "failed"]

### Verify that all notebooks have been run etc

In [51]:
qc_files = sorted((project / "Dataframe_Saves").glob("qc_*.csv"))

In [52]:
assert qc_files, "no qc_*.csv yet - run the loop first"

In [53]:
qc = pd.concat([pd.read_csv(f) for f in qc_files])

In [54]:
schemas = {tuple(pd.read_csv(f, nrows=0).columns) for f in qc_files}

In [55]:
assert len(schemas) == 1, f"{len(schemas)} different qc schemas — stale files from an older run"

In [56]:
assert len(qc_files) == len(sample_ids), f"{len(qc_files)} qc files for {len(sample_ids)} samples"

In [57]:
qc = qc.sort_values(["condition", "sample"]).reset_index(drop=True)

In [58]:
qc.to_csv(project / "metadata" / "qc_summary.csv", index=False)

In [59]:
len(qc), len(sample_ids)

(45, 45)

In [60]:
qc.n_cells_final.sum()

np.int64(238187)

In [61]:
(
    qc.groupby("condition")[["n_cells_final", "miqc_mt_cutoff", "pct_doublets"]]
    .agg(["sum", "median"]).round(1).to_string()
)

'          n_cells_final         miqc_mt_cutoff        pct_doublets       \n                    sum  median            sum median          sum median\ncondition                                                                \nER                96505  5442.0          297.9   12.4        152.6    8.3\nHER2              38352  4815.0          127.6   19.1         53.1    8.0\nN                 51208  4062.0          194.7   10.7         89.9    6.9\nTN                52122  4971.5          147.7   12.0         71.2    8.8'

In [62]:
qc

,sample,condition,n_cells_final,n_genes,miqc_mt_cutoff,pct_doublets,pc1_depth_corr,median_counts,median_genes,ror1_egfr_dp,ror1_egfr_dp_doublet_pct,epcam_erbb2_dp,epcam_erbb2_dp_doublet_pct,miqc_converged,mt_filter_path,mt_floor
0,GSM4909295_ER-AH0319,ER,2870,33538,11.4,5.8,-0.701,2148,869,16,0.0,264,5.7,True,miqc,10
1,GSM4909296_ER-MH0001,ER,6279,33538,10.0,10.4,-0.609,1341,533,0,NaN,22,22.7,True,miqc,10
2,GSM4909297_ER-MH0125,ER,6411,33538,10.0,11.0,0.695,954,522,3,0.0,357,19.3,True,miqc,10
3,GSM4909298_ER-PM0360,ER,5486,33538,18.2,8.3,0.712,1210,537,1,0.0,218,9.6,True,miqc,10
4,GSM4909299_ER-MH0114-T3,ER,7595,33538,10.0,10.8,0.520,2765,1115,9,22.2,1304,21.0,True,miqc,10
5,GSM4909300_ER-MH0032,ER,1229,33538,11.1,4.1,0.572,2192,864,1,0.0,97,5.2,True,miqc,10
6,GSM4909301_ER-MH0042,ER,4720,33538,10.0,8.3,0.489,1665,765,0,NaN,99,14.1,True,miqc,10
7,GSM4909302_ER-MH0025,ER,8237,33538,11.1,11.2,-0.576,1725,807,1,0.0,385,23.1,True,miqc,10
8,GSM4909303_ER-MH0151,ER,2601,33538,28.6,5.4,0.728,2220,968,22,4.5,350,4.3,True,miqc,10
9,GSM4909304_ER-MH0163,ER,5935,33538,31.3,8.2,0.716,3964,1432,0,NaN,371,14.6,True,miqc,10


In [63]:
for _, r in qc.iterrows():
    notes = []
    if abs(r.pc1_depth_corr) > 0.9: notes.append(f"PC1~depth {r.pc1_depth_corr}")
    if r.pct_doublets > 12:     notes.append(f"doublets {r.pct_doublets}%")
    if r.miqc_mt_cutoff > 50:   notes.append(f"miQC cutoff {r.miqc_mt_cutoff}% - model probably failed")
    if r.n_cells_final < 500:   notes.append(f"only {r.n_cells_final} cells")
    if notes:
        print(f"{r['sample']:32s} {'; '.join(notes)}")

GSM4909317_ER-MH0173-T           doublets 12.5%
GSM4909290_HER2-PM0337           doublets 12.6%
GSM4909294_HER2-MH0176           doublets 14.8%
GSM4909282_TN-MH0135             doublets 12.2%
GSM4909288_TN-B1-MH0177          doublets 15.9%


In [64]:
rows = []

In [65]:
for patient in sample_ids:
    path = head_dir / f"{patient}.h5ad"
    if not path.exists():
        rows.append({"sample": patient, "written": False}); continue
    a = ad.read_h5ad(path, backed="r")
    rows.append({"sample": patient, "written": True, "cells": a.n_obs, "genes": a.n_vars,
                 **{g: g in a.var_names for g in ["ROR1", "ROR2", "EGFR", "ERBB2", "EPCAM"]}})
    a.file.close()

In [66]:
out = pd.DataFrame(rows)

In [67]:
out.written.sum(), len(sample_ids)

(np.int64(45), 45)

In [68]:
lost = out[out.written & ~out[["ROR1", "ROR2", "EGFR", "ERBB2", "EPCAM"]].all(axis=1)]

In [69]:
lost["sample"].tolist()

[]

In [70]:
assert out[out.written].genes.nunique() == 1, "gene sets differ between samples"